# Experiment 1 · Bearing-fault detector

**Question:** can a model predict whether a bearing is *healthy* or *damaged* from motor phase currents?

**Protocol (locked in exploration):**
- 6 bearings held out **first** and never touched until the final demo (2 healthy `K001,K006`, 2 artificial `KA01,KA15`, 2 real `KI01,KI16`).
- The remaining 26 bearings train 6 **independent** models (one per ML family, no ensembles).
- Each model is scored by **bearing-wise 5-fold CV** (identical folds for all models), repeated over 2 split seeds.
- The winner is chosen on **mean per-condition bearing-AUROC** (the honest evaluation unit), then calibrated
  and given a decision threshold targeting **≤ 10 % healthy false-alarms**.
- Finally the 6 held-out bearings are run through the finished pipeline as a presentation demo.

In [ ]:
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'pyproject.toml').exists():
        sys.path.insert(0, str(_p)); break
from src import experiments as ex
import numpy as np, pandas as pd

df, X = ex.load_data()          # all 76,868 windows + 36 MCSA features
cache = ex.load_cache()          # decimated raw current windows for the CNN
print(f"windows: {len(df):,} | features: {X.shape[1]} | bearings: {df.bearing.nunique()}")

### Hold-out split (the demo set, kept pure)

In [ ]:
HOLDOUT = ["K001", "K006", "KA01", "KA15", "KI01", "KI16"]  # 2 healthy + 2 artificial + 2 real
ho = df.bearing.isin(HOLDOUT).to_numpy()
train_df, X_train = df[~ho].reset_index(drop=True), X[~ho]
demo_df,  X_demo  = df[ho].reset_index(drop=True),  X[ho]
print(f"training: {train_df.bearing.nunique()} bearings, {len(train_df):,} windows")
print(f"demo    : {len(HOLDOUT)} held-out bearings -> {', '.join(sorted(HOLDOUT))}")

### Model comparison — bearing-wise 5-fold CV, mean ± std over 2 seeds

Each row is one independent model (logreg / xgboost / random forest / RBF-kernel
approximation / MLP on features / 1D-CNN on raw windows). The CNN trains on
every-2nd overlapping window (they are 50 %-overlapping near-repeats).

In [ ]:
SEEDS = (42, 7)
cv_table, oof = ex.cv_summary(train_df, X_train, cache, seeds=SEEDS, n_folds=5)

In [ ]:
pd.set_option("display.width", 220)

In [ ]:
best, ranked = ex.best_model(cv_table)
print(f"winner (bearing-AUROC): {best}")
display(ranked[["model", "bearing_auroc±", "bearing_auroc±std", "window_auroc±",
                "auroc_N15_M01_F10±", "auroc_N09_M07_F10±"]])

### Final model + calibration + threshold (tuned on out-of-fold predictions only)

In [ ]:
model, kind = ex.fit_final(train_df, X_train, cache, best)
print(f"final fit: {best}  (kind={kind})")

y_oof, p_oof = oof[best]
p_demo = ex.predict(demo_df, X_demo, cache, model, best)
p_demo_cal, threshold = ex.calibrate_and_threshold(y_oof, p_oof, demo_df["label"], p_demo)
print(f"decision threshold (healthy false-alarm <= 10%): {threshold:.3f}")

### Demo — the held-out bearings predict healthy vs damaged (per file)

In [ ]:
verdicts = ex.recording_verdicts(demo_df, p_demo_cal, threshold)
display(verdicts[["bearing", "condition", "label", "n_windows", "p_damaged", "verdict"]])
acc = (verdicts["verdict"] == verdicts["label"]).mean()
print(f"demo verdict accuracy: {acc:.0%}  ({verdicts['verdict'].eq(verdicts['label']).sum()}/{len(verdicts)} files)")

In [ ]:
from utils.metrics import threshold_scores
display(pd.Series(threshold_scores(demo_df["label"], p_demo_cal, threshold), name="demo (held-out)"))

### Save artifacts

In [ ]:
import json, joblib
from pipeline.config import MODEL_DIR, RESULTS_DIR
MODEL_DIR.mkdir(parents=True, exist_ok=True); RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ext = "joblib" if kind == "sklearn" else "keras"
model_file = MODEL_DIR / f"exp1_{best}.{ext}"
if kind == "sklearn":
    joblib.dump(model, model_file)
else:
    model.save(model_file)

config = {
    "model": best, "backend": "features" if kind == "sklearn" else "raw",
    "threshold": threshold, "window": 16384, "step": 8192,
    "feature_cols": ex.feature_cols(train_df),
}
(MODEL_DIR / "exp1_config.json").write_text(json.dumps(config, indent=2, default=str))
cv_table.to_csv(RESULTS_DIR / "exp1_cv_table.csv", index=False)
verdicts.to_csv(RESULTS_DIR / "exp1_demo_verdicts.csv", index=False)
print("saved:", model_file, "|", MODEL_DIR / "exp1_config.json")

### Findings *updated after execution*